#Librerias

In [1]:
!pip install catboost
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 6.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import os

# Librerías de manipulación de datos
import pandas as pd
import numpy as np

# Opciones de Pandas
pd.set_option('display.max_columns', None)

# Librerías de visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import plotly.express as px

# Librerías para estadística y pruebas
import scipy.stats as stats
from scipy.stats import gaussian_kde, kstest, norm, shapiro
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Librerías de machine learning y preprocesamiento
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Librerías para métricas de modelos
from sklearn.model_selection import train_test_split, GridSearchCV
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error,  mean_absolute_error, mean_absolute_percentage_error, r2_score, classification_report
import math
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier

#Carga de datos

In [3]:
os.environ['KAGGLE_CONFIG_DIR'] = '.'

In [4]:
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia

  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 342MB/s]


In [5]:
!unzip udea-ai-4-eng-20252-pruebas-saber-pro-colombia

Archive:  udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
  inflating: submission_example.csv  
  inflating: test.csv                
  inflating: train.csv               


In [6]:
df_train=pd.read_csv('train.csv')

In [7]:
df_train.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,Si,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,No,N,No,Si,No,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,No,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,No,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,Si,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


In [8]:
df_test1=pd.read_csv('test.csv')
df_test1.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,550236,20183,TRABAJO SOCIAL,BOLIVAR,Menos de 500 mil,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica completa,Si,No,N,Si,Si,Si,Primaria completa,0.328,0.219,0.317,0.247
1,98545,20203,ADMINISTRACION COMERCIAL Y DE MERCADEO,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 2,Si,Secundaria (Bachillerato) completa,Si,No,N,No,Si,Si,Técnica o tecnológica completa,0.227,0.283,0.296,0.324
2,499179,20212,INGENIERIA MECATRONICA,BOGOTÁ,Entre 1 millón y menos de 2.5 millones,0,Estrato 3,Si,Secundaria (Bachillerato) incompleta,Si,No,N,No,Si,Si,Secundaria (Bachillerato) completa,0.285,0.228,0.294,0.247
3,782980,20195,CONTADURIA PUBLICA,SUCRE,Entre 1 millón y menos de 2.5 millones,Entre 21 y 30 horas,Estrato 1,No,Primaria incompleta,Si,No,N,No,No,No,Primaria incompleta,0.160,0.408,0.217,0.294
4,785185,20212,ADMINISTRACION DE EMPRESAS,ATLANTICO,Entre 2.5 millones y menos de 4 millones,Entre 11 y 20 horas,Estrato 2,Si,Secundaria (Bachillerato) completa,Si,No,N,No,Si,Si,Secundaria (Bachillerato) completa,0.209,0.283,0.306,0.286


#Preprocesamiento para train

## Datos faltantes

En el análisis exploratorio se encontró que el dataset de entrenamiento está conformado por 692500 filas y 21 columnas (variables). El dataset no tiene columnas repetidas con el mismo nombre pero si tiene repetida la columna "F_TIENEINTERNET" como "F_TIENEINTERNET.1" esto debido a que ambas columnas tienen la misma cantidad de datos con SI y NO y tambien la misma cantidad de faltantes. No tiene filas duplicadas pero si tiene datos faltantes; entre las columnas con mayor proporción de faltantes se encuentran 'F_TIENEAUTOMOVIL', 'F_TIENELAVADORA' y 'F_TIENE COMPUTADOR' que representan más del 50% de los faltantes otras como 'E_VALORMATRICULAUNIVERSIDAD' y 'E_PAGOMATRICULAPROPIO' no representan ni el 1% de los faltantes. Se modificaron los tipos de datos para 'ID' y para 'PERIODO_ACADEMICO' puesto que estas variables aunque sean números están expresando categorías. El dataset solo tiene 4 variables númericas que son 'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4'.

In [9]:
df_train= df_train.drop(columns=['F_TIENEINTERNET.1'],  errors='ignore') #Eliminación de columna repetida

In [10]:
df_train['ID']=df_train['ID'].astype('object')
df_train['PERIODO_ACADEMICO']=df_train['PERIODO_ACADEMICO'].astype('object')

Algunas variables no son ordinales como:

'F_TIENEINTERNET','F_TIENELAVADORA','F_TIENEAUTOMOVIL','F_TIENECOMPUTADOR','E_PAGOMATRICULAPROPIO',

es decir, sus categorias no representan un orden, los valores faltantes de estas serán reemplazados con la categoria 'otro'. La variable 'F_ESTRATOVIVIENDA' tiene una categoria llamada 'Sin Estrato' aquí se añadirán todos los valores faltantes de dicha variable. Las variables restantes serán tratadas posteriormente luego de convertirlas a categoricas para ayudar al modelo a interpretar la ordinalidad de cada una.

In [11]:
for col in ['F_TIENEINTERNET', 'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'F_TIENECOMPUTADOR','E_PAGOMATRICULAPROPIO']:
    df_train[col] = df_train[col].fillna('otro')

In [12]:
df_train['F_ESTRATOVIVIENDA']=df_train['F_ESTRATOVIVIENDA'].fillna('Sin Estrato')

In [13]:
df_train = df_train.drop(columns=['ID'], errors='ignore')

## Feature engineering

### Creación de variables

**CATEGORIA_PROGRAMA**

Esta variable agrupa los distintos programas académicos (más de 900 originalmente) en categorías temáticas o áreas de conocimiento, como Ingenierías, Ciencias Sociales, Administración y Economía, Educación, Ciencias de la Salud, Artes y Comunicación, entre otras. Su objetivo es representar el campo disciplinar de formación del estudiante sin perder información relevante, pero reduciendo la alta cardinalidad original. Esto facilita al modelo identificar patrones de desempeño por área académica.

In [14]:
def categorizar_programa(x):
    x = str(x).upper()

    if any(word in x for word in ['INGENIER', 'SISTEMAS', 'SOFTWARE', 'COMPUTACIÓN', 'ELECTR', 'INDUSTRIAL', 'CIVIL', 'MECANIC', 'TECNOLOG','ARQUI','COMPU','URBAN','CONSTR','OCEANO','DISENO']):
        return 'INGENIERÍAS Y CIENCIAS APLICADAS'
    elif any(word in x for word in ['MEDIC', 'ENFERM', 'SALUD', 'FISIOTERAP', 'BIOQUIM', 'NUTRIC', 'ODONTO','INTRU','INSTRU','BACTER','MICROB','OPTO','TERAPIA','FONO','FARMACE','FARMA','ENFERM','VETERIN']):
        return 'CIENCIAS DE LA SALUD'
    elif any(word in x for word in ['SOCIOLOG', 'PSICOLOG', 'FILOSOF', 'HISTOR', 'ANTROPOLOG', 'TEOLOG', 'BIBLIC', 'POLITIC', 'GOBIERNO','DERECHO','TRABAJO','POLÍTICA','PSICÓLOGA','INTERNACIONALES','JURIS','ARQUEO','ARCHI','GERONT']):
        return 'CIENCIAS SOCIALES Y HUMANAS'
    elif any(word in x for word in ['ADMINISTR', 'ECONOM', 'NEGOC', 'FINANZ', 'COMERC', 'EMPRES', 'MARKETING', 'GERENCIA','CONTAD']):
        return 'ADMINISTRACIÓN Y ECONOMÍA'
    elif any(word in x for word in ['LICENCIAT', 'PEDAGOG', 'DOCEN', 'EDUCAC', 'ENSEÑAN','TRADUCC','LINGUISTICA']):
        return 'EDUCACIÓN'
    elif any(word in x for word in ['MATEM', 'FISIC', 'QUIMIC', 'BIOLOG', 'GEOLOG', 'ECOLOG', 'ASTRONOM','FÍSICA','QUÍMICA','ESTADISTICA','ESTADÍSTICA','ZOO']):
        return 'CIENCIAS EXACTAS Y NATURALES'
    elif any(word in x for word in ['ARTE', 'MUSICA', 'TEATRO', 'CINE', 'DISEÑ', 'PUBLICID', 'COMUNIC', 'LITERAT', 'FILOL', 'LENGUA','MÚSICA','DANZA','BANDA','LITERARIA','LITERARIOS','FOTOGRAFÍA','AUDIOVISUALES','PRODUCCIÓN','ANIMACIÓN','CULTURAL']):
        return 'ARTES, COMUNICACIÓN Y HUMANIDADES'
    elif any(word in x for word in ['TURIS', 'HOTEL', 'GASTRON', 'CULINAR']):
        return 'TURISMO Y HOTELERÍA'
    elif any(word in x for word in ['CRIMINAL', 'INVESTIGAC', 'POLIC', 'MILITAR', 'NAVA', 'SEGURIDAD']):
        return 'CRIMINALÍSTICA Y SEGURIDAD'
    elif any(word in x for word in ['DEPORTE', 'DEPORTIVO','DEPORTIVA']):
        return 'DEPORTE'
    elif any(word in x for word in ['MERCADEO', 'MERCADOLOGIA','PLANEACION']):
        return 'NEGOCIOS Y MERCADEO'
    else:
        return 'OTRAS'

# Aplicar la función
df_train['CATEGORIA_PROGRAMA'] = df_train['E_PRGM_ACADEMICO'].apply(categorizar_programa)


In [15]:
df_train['CATEGORIA_PROGRAMA'].unique()

array(['CIENCIAS DE LA SALUD', 'CIENCIAS SOCIALES Y HUMANAS',
       'ARTES, COMUNICACIÓN Y HUMANIDADES', 'ADMINISTRACIÓN Y ECONOMÍA',
       'INGENIERÍAS Y CIENCIAS APLICADAS', 'TURISMO Y HOTELERÍA',
       'EDUCACIÓN', 'CIENCIAS EXACTAS Y NATURALES', 'DEPORTE',
       'NEGOCIOS Y MERCADEO', 'CRIMINALÍSTICA Y SEGURIDAD', 'OTRAS'],
      dtype=object)

**REGION**

Esta variable clasifica a los estudiantes según la región geográfica en la que se encuentra su programa académico. A partir del departamento (E_PRGM_DEPARTAMENTO), se agruparon los territorios de Colombia en grandes regiones: Andina, Caribe, Pacífica, Orinoquía y Amazónica. Esta agrupación permite capturar diferencias contextuales en la calidad de la educación, acceso a recursos o infraestructura entre zonas del país, las cuales pueden influir significativamente en el rendimiento académico.

In [16]:
region_map = {
    'ANTIOQUIA': 'Andina', 'CUNDINAMARCA': 'Andina', 'BOGOTÁ': 'Andina',
    'BOYACA': 'Andina', 'SANTANDER': 'Andina', 'NORTE SANTANDER': 'Andina',
    'CALDAS': 'Andina', 'QUINDIO': 'Andina', 'RISARALDA': 'Andina', 'TOLIMA': 'Andina',
    'HUILA': 'Andina',
    'ATLANTICO': 'Caribe', 'BOLIVAR': 'Caribe', 'CESAR': 'Caribe', 'CORDOBA': 'Caribe',
    'LA GUAJIRA': 'Caribe', 'MAGDALENA': 'Caribe', 'SUCRE': 'Caribe', 'SAN ANDRES': 'Caribe',
    'VALLE': 'Pacífica', 'CAUCA': 'Pacífica', 'NARIÑO': 'Pacífica', 'CHOCO': 'Pacífica',
    'META': 'Orinoquía', 'CASANARE': 'Orinoquía', 'ARAUCA': 'Orinoquía',
    'AMAZONAS': 'Amazónica', 'PUTUMAYO': 'Amazónica', 'CAQUETA': 'Amazónica',
    'GUAVIARE': 'Amazónica', 'VAUPES': 'Amazónica'
}

df_train['REGION'] = df_train['E_PRGM_DEPARTAMENTO'].map(region_map)


**AÑO**

Esta variable representa el año de presentación o cohorte académica del estudiante, extraído del campo PERIODO_ACADEMICO. Captura el componente temporal del dataset y permite observar posibles tendencias o cambios en el desempeño a lo largo del tiempo, relacionados con políticas educativas, reformas curriculares o variaciones en la cobertura del examen

In [17]:
df_train['AÑO'] = df_train['PERIODO_ACADEMICO'].astype(str).str[:4].astype(int)

**INDICE_SOCIOECO**

El índice socioeconómico es un indicador compuesto que resume el nivel de bienestar del hogar del estudiante. Se calculó sumando variables binarias sobre la tenencia de bienes y servicios básicos (F_TIENEINTERNET, F_TIENECOMPUTADOR, F_TIENEAUTOMOVIL, F_TIENELAVADORA). Un valor de 4 refleja un entorno con mayor acceso a recursos tecnológicos y económicos. Esta variable permite representar de manera compacta el contexto socioeconómico, fuertemente correlacionado con el rendimiento educativo.

In [18]:
df_train['INDICE_SOCIOECO'] = (
    (df_train['F_TIENEINTERNET'] == 'Si').astype(int) +
    (df_train['F_TIENECOMPUTADOR'] == 'Si').astype(int) +
    (df_train['F_TIENEAUTOMOVIL'] == 'Si').astype(int) +
    (df_train['F_TIENELAVADORA'] == 'Si').astype(int)
)

**VALOR_MATRICULA_NUM**

Corresponde a una versión numérica del rango de valor de matrícula (E_VALORMATRICULAUNIVERSIDAD), expresada mediante el punto medio de cada intervalo monetario. Por ejemplo, “Entre 1 millón y menos de 2.5 millones” se convierte en 1,750,000. Esta transformación permite utilizar la variable como un indicador cuantitativo del nivel económico institucional o de acceso financiero del estudiante, asociado al poder adquisitivo y a los recursos disponibles durante su formación.

In [19]:
matricula_map = {
    'Menos de 500 mil': 250_000,
    'Entre 500 mil y menos de 1 millón': 750_000,
    'Entre 1 millón y menos de 2.5 millones': 1_750_000,
    'Entre 2.5 millones y menos de 4 millones': 3_250_000,
    'Entre 4 millones y menos de 5.5 millones': 4_750_000,
    'Entre 5.5 millones y menos de 7 millones': 6_250_000,
    'Más de 7 millones': 8_000_000,
    'No pagó matrícula': 0,
}

df_train['VALOR_MATRICULA_NUM'] = df_train['E_VALORMATRICULAUNIVERSIDAD'].map(matricula_map)
df_train['VALOR_MATRICULA_NUM']=df_train['VALOR_MATRICULA_NUM'].fillna(df_train['VALOR_MATRICULA_NUM'].median())

**EDU_PADRE_NUM Y EDU_MADRE_NUM**

Esta variable convierte los niveles educativos del padre (F_EDUCACIONPADRE) en una escala numérica ordinal de 0 a 9, donde 0 indica “Ninguno” y 9 “Postgrado”. Permite representar la educación paterna en un formato que conserva el orden jerárquico y facilita su interpretación en modelos predictivos. Esta información es clave para comprender el entorno educativo familiar. De forma análoga, EDU_MADRE_NUM representa el nivel educativo de la madre en escala ordinal. Junto con EDU_PADRE_NUM, permite identificar diferencias en el nivel de formación parental, que pueden influir en el acompañamiento académico del estudiante. Estas dos variables son esenciales para estimar el efecto del capital cultural y educativo del hogar en el desempeño global.

In [20]:
edu_map = {
    'Ninguno': 0,
    'Primaria incompleta': 1,
    'Primaria completa': 2,
    'Secundaria (Bachillerato) incompleta': 3,
    'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5,
    'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7,
    'Educación profesional completa': 8,
    'Postgrado': 9,
    'No sabe': 0,
    'No Aplica': 0,
}

df_train['EDU_PADRE_NUM'] = df_train['F_EDUCACIONPADRE'].map(edu_map)
df_train['EDU_MADRE_NUM'] = df_train['F_EDUCACIONMADRE'].map(edu_map)

moda_padre = df_train['EDU_PADRE_NUM'].mode()[0]
moda_madre = df_train['EDU_MADRE_NUM'].mode()[0]

df_train['EDU_PADRE_NUM']=df_train['EDU_PADRE_NUM'].fillna(moda_padre)
df_train['EDU_MADRE_NUM']=df_train['EDU_MADRE_NUM'].fillna(moda_madre)

**EDU_PROMEDIO_FAM**

El promedio educativo familiar se obtuvo calculando la media entre los niveles educativos del padre y la madre (EDU_PADRE_NUM y EDU_MADRE_NUM). Esta variable cuantifica el capital educativo del hogar y funciona como un indicador del entorno formativo y cultural del estudiante. Diversos estudios muestran que un mayor nivel educativo de los padres está positivamente asociado al desempeño académico de los hijos.

**ALGUNO_SUPERIOR**

Es una variable binaria (0 o 1) que indica si al menos uno de los padres posee educación superior (técnica, tecnológica o profesional). Se construyó a partir de los valores numéricos de educación del padre y la madre. Este indicador captura la exposición del estudiante a un entorno familiar con experiencia universitaria, lo cual suele influir en la motivación, hábitos de estudio y apoyo académico disponible.

In [21]:
df_train['EDU_PROMEDIO_FAM'] = df_train[['EDU_PADRE_NUM', 'EDU_MADRE_NUM']].mean(axis=1)

df_train['ALGUNO_SUPERIOR'] = np.where(
    (df_train['EDU_PADRE_NUM'] >= 6) | (df_train['EDU_MADRE_NUM'] >= 6), 1, 0
)

**HORAS_TRABAJO_NUM**

Representa el número estimado de horas de trabajo semanal del estudiante, transformando las categorías de E_HORASSEMANATRABAJA (“Entre 11 y 20 horas”, “Más de 30 horas”, etc.) en valores numéricos ordinales aproximados (por ejemplo, 0, 5, 15, 25, 35). Esta variable refleja el grado de carga laboral del estudiante, que puede afectar su dedicación al estudio y, por tanto, su rendimiento en las pruebas.

In [22]:
horas_map = {
    '0': 0,
    'Menos de 10 horas': 5,
    'Entre 11 y 20 horas': 15,
    'Entre 21 y 30 horas': 25,
    'Más de 30 horas': 35
}

df_train['HORAS_TRABAJO_NUM'] = df_train['E_HORASSEMANATRABAJA'].map(horas_map)
moda_horas = df_train['HORAS_TRABAJO_NUM'].mode()[0]
df_train['HORAS_TRABAJO_NUM']=df_train['HORAS_TRABAJO_NUM'].fillna(moda_horas)

## Eliminación de variables

In [23]:
df_clean=df_train.copy()

In [24]:
cols_drop = [
    'PERIODO_ACADEMICO',
    'E_PRGM_ACADEMICO',
    'E_PRGM_DEPARTAMENTO',
    'E_VALORMATRICULAUNIVERSIDAD',
    'E_HORASSEMANATRABAJA',
    'F_TIENEINTERNET',
    'F_EDUCACIONPADRE',
    'F_TIENELAVADORA',
    'F_TIENEAUTOMOVIL',
    'F_TIENECOMPUTADOR',
    'F_EDUCACIONMADRE',
    'E_PRIVADO_LIBERTAD'
]
df_model = df_clean.drop(columns=cols_drop, errors='ignore')


In [25]:
df_model_2 = pd.get_dummies(df_model, columns=['AÑO','VALOR_MATRICULA_NUM', 'INDICE_SOCIOECO','REGION', 'F_ESTRATOVIVIENDA','E_PAGOMATRICULAPROPIO', 'ALGUNO_SUPERIOR',
    'EDU_PADRE_NUM', 'EDU_MADRE_NUM', 'EDU_PROMEDIO_FAM','HORAS_TRABAJO_NUM','CATEGORIA_PROGRAMA'], drop_first=True,dtype=int)

In [26]:
from sklearn.preprocessing import LabelEncoder
# Definición del orden correcto
mapeo_ordenado = {
    'alto': 0,
    'medio-alto': 1,
    'medio-bajo': 2,
    'bajo': 3
}
df_model_2['RENDIMIENTO_GLOBAL_encoded'] = df_model_2['RENDIMIENTO_GLOBAL'].map(mapeo_ordenado)

#Preprocesamiento para test

## Datos faltantes

Se evalua que el conjunto de datos para test tenga las mismas variables y los mismos faltantes que el conjunto de datos de train para aplicar el mismo preprocesamiento

In [29]:
df_test=df_test1.copy()

In [30]:
def visual_check(df):

    print('Shape:\n',df.shape,'\n')

    print('Data Types:\n', df.dtypes,'\n\n')

    df_colum_duplicated=df.columns.duplicated()
    print("Columnas duplicadas:\n",df_colum_duplicated,'\n\n')

    df_fil_duplicated=df.duplicated()
    print("Filas duplicadas:\n",df_fil_duplicated,'\n\n')

    df_colum_faltantes=df.columns[df.isnull().any()]
    print("Datos faltantes por columnas:\n",df_colum_faltantes,'\n\n')

    display(df.describe().T)

visual_check(df_test)

Shape:
 (296786, 20) 

Data Types:
 ID                               int64
PERIODO_ACADEMICO                int64
E_PRGM_ACADEMICO                object
E_PRGM_DEPARTAMENTO             object
E_VALORMATRICULAUNIVERSIDAD     object
E_HORASSEMANATRABAJA            object
F_ESTRATOVIVIENDA               object
F_TIENEINTERNET                 object
F_EDUCACIONPADRE                object
F_TIENELAVADORA                 object
F_TIENEAUTOMOVIL                object
E_PRIVADO_LIBERTAD              object
E_PAGOMATRICULAPROPIO           object
F_TIENECOMPUTADOR               object
F_TIENEINTERNET.1               object
F_EDUCACIONMADRE                object
INDICADOR_1                    float64
INDICADOR_2                    float64
INDICADOR_3                    float64
INDICADOR_4                    float64
dtype: object 


Columnas duplicadas:
 [False False False False False False False False False False False False
 False False False False False False False False] 


Filas duplicadas:
 

,count,mean,std,min,25%,50%,75%,max
ID,296786.0,494730.695238,285576.351746,2.0,247318.250,494808.500,742441.000,989285.000
PERIODO_ACADEMICO,296786.0,20198.379078,10.525743,20183.0,20195.000,20195.000,20203.000,20213.000
INDICADOR_1,296786.0,0.268235,0.121352,0.0,0.205,0.242,0.312,0.663
INDICADOR_2,296786.0,0.259989,0.094596,0.0,0.214,0.269,0.306,0.484
INDICADOR_3,296786.0,0.262855,0.059635,0.0,0.254,0.278,0.295,0.322
INDICADOR_4,296786.0,0.262450,0.067587,0.0,0.253,0.284,0.302,0.331


In [31]:
# Seleccionar solo las columnas de interés
cols = [
    'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA',
       'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE',
       'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PAGOMATRICULAPROPIO',
       'F_TIENECOMPUTADOR', 'F_TIENEINTERNET.1', 'F_EDUCACIONMADRE'
]

# Crear dataset de faltantes
faltantes_df = pd.DataFrame({
    "Columna": cols,
    "Faltantes": df_test[cols].isnull().sum().values,
    "Porcentaje (%)": (df_test[cols].isnull().mean().values * 100).round(2)
})

faltantes_df

,Columna,Faltantes,Porcentaje (%)
0,E_VALORMATRICULAUNIVERSIDAD,2723,0.92
1,E_HORASSEMANATRABAJA,13379,4.51
2,F_ESTRATOVIVIENDA,13795,4.65
3,F_TIENEINTERNET,11539,3.89
4,F_EDUCACIONPADRE,9993,3.37
5,F_TIENELAVADORA,17259,5.82
6,F_TIENEAUTOMOVIL,18918,6.37
7,E_PAGOMATRICULAPROPIO,2807,0.95
8,F_TIENECOMPUTADOR,16439,5.54
9,F_TIENEINTERNET.1,11539,3.89


Como tenemos las mismas variables con datos faltantes, aplicamos los mismos métodos de preprocesamiento y creamos las mismas nuevas variables creadas en test

In [32]:
df_test= df_test.drop(columns=['F_TIENEINTERNET.1'],  errors='ignore') #Eliminación de columna repetida

In [33]:
df_test['ID']=df_test['ID'].astype('object')
df_test['PERIODO_ACADEMICO']=df_test['PERIODO_ACADEMICO'].astype('object')

In [34]:
for col in ['F_TIENEINTERNET', 'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'F_TIENECOMPUTADOR','E_PAGOMATRICULAPROPIO']:
    df_test[col] = df_test[col].fillna('otro')

In [35]:
df_test['F_ESTRATOVIVIENDA']=df_test['F_ESTRATOVIVIENDA'].fillna('Sin Estrato')

In [36]:
df_test = df_test.drop(columns=['ID'], errors='ignore')

## Feature engineering

### Creación de variables

**CATEGORIA_PROGRAMA**

Esta variable agrupa los distintos programas académicos (más de 900 originalmente) en categorías temáticas o áreas de conocimiento, como Ingenierías, Ciencias Sociales, Administración y Economía, Educación, Ciencias de la Salud, Artes y Comunicación, entre otras. Su objetivo es representar el campo disciplinar de formación del estudiante sin perder información relevante, pero reduciendo la alta cardinalidad original. Esto facilita al modelo identificar patrones de desempeño por área académica.

In [37]:
def categorizar_programa(x):
    x = str(x).upper()

    if any(word in x for word in ['INGENIER', 'SISTEMAS', 'SOFTWARE', 'COMPUTACIÓN', 'ELECTR', 'INDUSTRIAL', 'CIVIL', 'MECANIC', 'TECNOLOG','ARQUI','COMPU','URBAN','CONSTR','OCEANO','DISENO']):
        return 'INGENIERÍAS Y CIENCIAS APLICADAS'
    elif any(word in x for word in ['MEDIC', 'ENFERM', 'SALUD', 'FISIOTERAP', 'BIOQUIM', 'NUTRIC', 'ODONTO','INTRU','INSTRU','BACTER','MICROB','OPTO','TERAPIA','FONO','FARMACE','FARMA','ENFERM','VETERIN']):
        return 'CIENCIAS DE LA SALUD'
    elif any(word in x for word in ['SOCIOLOG', 'PSICOLOG', 'FILOSOF', 'HISTOR', 'ANTROPOLOG', 'TEOLOG', 'BIBLIC', 'POLITIC', 'GOBIERNO','DERECHO','TRABAJO','POLÍTICA','PSICÓLOGA','INTERNACIONALES','JURIS','ARQUEO','ARCHI','GERONT']):
        return 'CIENCIAS SOCIALES Y HUMANAS'
    elif any(word in x for word in ['ADMINISTR', 'ECONOM', 'NEGOC', 'FINANZ', 'COMERC', 'EMPRES', 'MARKETING', 'GERENCIA','CONTAD']):
        return 'ADMINISTRACIÓN Y ECONOMÍA'
    elif any(word in x for word in ['LICENCIAT', 'PEDAGOG', 'DOCEN', 'EDUCAC', 'ENSEÑAN','TRADUCC','LINGUISTICA']):
        return 'EDUCACIÓN'
    elif any(word in x for word in ['MATEM', 'FISIC', 'QUIMIC', 'BIOLOG', 'GEOLOG', 'ECOLOG', 'ASTRONOM','FÍSICA','QUÍMICA','ESTADISTICA','ESTADÍSTICA','ZOO']):
        return 'CIENCIAS EXACTAS Y NATURALES'
    elif any(word in x for word in ['ARTE', 'MUSICA', 'TEATRO', 'CINE', 'DISEÑ', 'PUBLICID', 'COMUNIC', 'LITERAT', 'FILOL', 'LENGUA','MÚSICA','DANZA','BANDA','LITERARIA','LITERARIOS','FOTOGRAFÍA','AUDIOVISUALES','PRODUCCIÓN','ANIMACIÓN','CULTURAL']):
        return 'ARTES, COMUNICACIÓN Y HUMANIDADES'
    elif any(word in x for word in ['TURIS', 'HOTEL', 'GASTRON', 'CULINAR']):
        return 'TURISMO Y HOTELERÍA'
    elif any(word in x for word in ['CRIMINAL', 'INVESTIGAC', 'POLIC', 'MILITAR', 'NAVA', 'SEGURIDAD']):
        return 'CRIMINALÍSTICA Y SEGURIDAD'
    elif any(word in x for word in ['DEPORTE', 'DEPORTIVO','DEPORTIVA']):
        return 'DEPORTE'
    elif any(word in x for word in ['MERCADEO', 'MERCADOLOGIA','PLANEACION']):
        return 'NEGOCIOS Y MERCADEO'
    else:
        return 'OTRAS'

# Aplicar la función
df_test['CATEGORIA_PROGRAMA'] = df_test['E_PRGM_ACADEMICO'].apply(categorizar_programa)


In [38]:
df_test['CATEGORIA_PROGRAMA'].unique()

array(['CIENCIAS SOCIALES Y HUMANAS', 'ADMINISTRACIÓN Y ECONOMÍA',
       'INGENIERÍAS Y CIENCIAS APLICADAS', 'CIENCIAS DE LA SALUD',
       'EDUCACIÓN', 'CRIMINALÍSTICA Y SEGURIDAD',
       'ARTES, COMUNICACIÓN Y HUMANIDADES', 'OTRAS',
       'NEGOCIOS Y MERCADEO', 'CIENCIAS EXACTAS Y NATURALES', 'DEPORTE',
       'TURISMO Y HOTELERÍA'], dtype=object)

**REGION**

Esta variable clasifica a los estudiantes según la región geográfica en la que se encuentra su programa académico. A partir del departamento (E_PRGM_DEPARTAMENTO), se agruparon los territorios de Colombia en grandes regiones: Andina, Caribe, Pacífica, Orinoquía y Amazónica. Esta agrupación permite capturar diferencias contextuales en la calidad de la educación, acceso a recursos o infraestructura entre zonas del país, las cuales pueden influir significativamente en el rendimiento académico.

In [39]:
region_map = {
    'ANTIOQUIA': 'Andina', 'CUNDINAMARCA': 'Andina', 'BOGOTÁ': 'Andina',
    'BOYACA': 'Andina', 'SANTANDER': 'Andina', 'NORTE SANTANDER': 'Andina',
    'CALDAS': 'Andina', 'QUINDIO': 'Andina', 'RISARALDA': 'Andina', 'TOLIMA': 'Andina',
    'HUILA': 'Andina',
    'ATLANTICO': 'Caribe', 'BOLIVAR': 'Caribe', 'CESAR': 'Caribe', 'CORDOBA': 'Caribe',
    'LA GUAJIRA': 'Caribe', 'MAGDALENA': 'Caribe', 'SUCRE': 'Caribe', 'SAN ANDRES': 'Caribe',
    'VALLE': 'Pacífica', 'CAUCA': 'Pacífica', 'NARIÑO': 'Pacífica', 'CHOCO': 'Pacífica',
    'META': 'Orinoquía', 'CASANARE': 'Orinoquía', 'ARAUCA': 'Orinoquía',
    'AMAZONAS': 'Amazónica', 'PUTUMAYO': 'Amazónica', 'CAQUETA': 'Amazónica',
    'GUAVIARE': 'Amazónica', 'VAUPES': 'Amazónica'
}

df_test['REGION'] = df_test['E_PRGM_DEPARTAMENTO'].map(region_map)


**AÑO**

Esta variable representa el año de presentación o cohorte académica del estudiante, extraído del campo PERIODO_ACADEMICO. Captura el componente temporal del dataset y permite observar posibles tendencias o cambios en el desempeño a lo largo del tiempo, relacionados con políticas educativas, reformas curriculares o variaciones en la cobertura del examen

In [40]:
df_test['AÑO'] = df_test['PERIODO_ACADEMICO'].astype(str).str[:4].astype(int)

**INDICE_SOCIOECO**

El índice socioeconómico es un indicador compuesto que resume el nivel de bienestar del hogar del estudiante. Se calculó sumando variables binarias sobre la tenencia de bienes y servicios básicos (F_TIENEINTERNET, F_TIENECOMPUTADOR, F_TIENEAUTOMOVIL, F_TIENELAVADORA). Un valor de 4 refleja un entorno con mayor acceso a recursos tecnológicos y económicos. Esta variable permite representar de manera compacta el contexto socioeconómico, fuertemente correlacionado con el rendimiento educativo.

In [41]:
df_test['INDICE_SOCIOECO'] = (
    (df_test['F_TIENEINTERNET'] == 'Si').astype(int) +
    (df_test['F_TIENECOMPUTADOR'] == 'Si').astype(int) +
    (df_test['F_TIENEAUTOMOVIL'] == 'Si').astype(int) +
    (df_test['F_TIENELAVADORA'] == 'Si').astype(int)
)

**VALOR_MATRICULA_NUM**

Corresponde a una versión numérica del rango de valor de matrícula (E_VALORMATRICULAUNIVERSIDAD), expresada mediante el punto medio de cada intervalo monetario. Por ejemplo, “Entre 1 millón y menos de 2.5 millones” se convierte en 1,750,000. Esta transformación permite utilizar la variable como un indicador cuantitativo del nivel económico institucional o de acceso financiero del estudiante, asociado al poder adquisitivo y a los recursos disponibles durante su formación.

In [42]:
matricula_map = {
    'Menos de 500 mil': 250_000,
    'Entre 500 mil y menos de 1 millón': 750_000,
    'Entre 1 millón y menos de 2.5 millones': 1_750_000,
    'Entre 2.5 millones y menos de 4 millones': 3_250_000,
    'Entre 4 millones y menos de 5.5 millones': 4_750_000,
    'Entre 5.5 millones y menos de 7 millones': 6_250_000,
    'Más de 7 millones': 8_000_000,
    'No pagó matrícula': 0,
}

df_test['VALOR_MATRICULA_NUM'] = df_test['E_VALORMATRICULAUNIVERSIDAD'].map(matricula_map)
df_test['VALOR_MATRICULA_NUM']=df_test['VALOR_MATRICULA_NUM'].fillna(df_test['VALOR_MATRICULA_NUM'].median())

**EDU_PADRE_NUM Y EDU_MADRE_NUM**

Esta variable convierte los niveles educativos del padre (F_EDUCACIONPADRE) en una escala numérica ordinal de 0 a 9, donde 0 indica “Ninguno” y 9 “Postgrado”. Permite representar la educación paterna en un formato que conserva el orden jerárquico y facilita su interpretación en modelos predictivos. Esta información es clave para comprender el entorno educativo familiar. De forma análoga, EDU_MADRE_NUM representa el nivel educativo de la madre en escala ordinal. Junto con EDU_PADRE_NUM, permite identificar diferencias en el nivel de formación parental, que pueden influir en el acompañamiento académico del estudiante. Estas dos variables son esenciales para estimar el efecto del capital cultural y educativo del hogar en el desempeño global.

In [43]:
edu_map = {
    'Ninguno': 0,
    'Primaria incompleta': 1,
    'Primaria completa': 2,
    'Secundaria (Bachillerato) incompleta': 3,
    'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5,
    'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7,
    'Educación profesional completa': 8,
    'Postgrado': 9,
    'No sabe': 0,
    'No Aplica': 0,
}

df_test['EDU_PADRE_NUM'] = df_test['F_EDUCACIONPADRE'].map(edu_map)
df_test['EDU_MADRE_NUM'] = df_test['F_EDUCACIONMADRE'].map(edu_map)

moda_padre = df_test['EDU_PADRE_NUM'].mode()[0]
moda_madre = df_test['EDU_MADRE_NUM'].mode()[0]

df_test['EDU_PADRE_NUM']=df_test['EDU_PADRE_NUM'].fillna(moda_padre)
df_test['EDU_MADRE_NUM']=df_test['EDU_MADRE_NUM'].fillna(moda_madre)

**EDU_PROMEDIO_FAM**

El promedio educativo familiar se obtuvo calculando la media entre los niveles educativos del padre y la madre (EDU_PADRE_NUM y EDU_MADRE_NUM). Esta variable cuantifica el capital educativo del hogar y funciona como un indicador del entorno formativo y cultural del estudiante. Diversos estudios muestran que un mayor nivel educativo de los padres está positivamente asociado al desempeño académico de los hijos.

**ALGUNO_SUPERIOR**

Es una variable binaria (0 o 1) que indica si al menos uno de los padres posee educación superior (técnica, tecnológica o profesional). Se construyó a partir de los valores numéricos de educación del padre y la madre. Este indicador captura la exposición del estudiante a un entorno familiar con experiencia universitaria, lo cual suele influir en la motivación, hábitos de estudio y apoyo académico disponible.

In [44]:
df_test['EDU_PROMEDIO_FAM'] = df_test[['EDU_PADRE_NUM', 'EDU_MADRE_NUM']].mean(axis=1)

df_test['ALGUNO_SUPERIOR'] = np.where(
    (df_test['EDU_PADRE_NUM'] >= 6) | (df_test['EDU_MADRE_NUM'] >= 6), 1, 0
)

**HORAS_TRABAJO_NUM**

Representa el número estimado de horas de trabajo semanal del estudiante, transformando las categorías de E_HORASSEMANATRABAJA (“Entre 11 y 20 horas”, “Más de 30 horas”, etc.) en valores numéricos ordinales aproximados (por ejemplo, 0, 5, 15, 25, 35). Esta variable refleja el grado de carga laboral del estudiante, que puede afectar su dedicación al estudio y, por tanto, su rendimiento en las pruebas.

In [45]:
horas_map = {
    '0': 0,
    'Menos de 10 horas': 5,
    'Entre 11 y 20 horas': 15,
    'Entre 21 y 30 horas': 25,
    'Más de 30 horas': 35
}

df_test['HORAS_TRABAJO_NUM'] = df_test['E_HORASSEMANATRABAJA'].map(horas_map)
moda_horas = df_test['HORAS_TRABAJO_NUM'].mode()[0]
df_test['HORAS_TRABAJO_NUM']=df_test['HORAS_TRABAJO_NUM'].fillna(moda_horas)

## Eliminación de variables

In [46]:
df_test_model=df_test.copy()

In [47]:
cols_drop = [
    'PERIODO_ACADEMICO',
    'E_PRGM_ACADEMICO',
    'E_PRGM_DEPARTAMENTO',
    'E_VALORMATRICULAUNIVERSIDAD',
    'E_HORASSEMANATRABAJA',
    'F_TIENEINTERNET',
    'F_EDUCACIONPADRE',
    'F_TIENELAVADORA',
    'F_TIENEAUTOMOVIL',
    'F_TIENECOMPUTADOR',
    'F_EDUCACIONMADRE',
    'E_PRIVADO_LIBERTAD'
]
df_test_model = df_test_model.drop(columns=cols_drop, errors='ignore')


In [48]:
df_test_model = pd.get_dummies(df_test_model, columns=['AÑO','VALOR_MATRICULA_NUM', 'INDICE_SOCIOECO','REGION', 'F_ESTRATOVIVIENDA','E_PAGOMATRICULAPROPIO', 'ALGUNO_SUPERIOR',
    'EDU_PADRE_NUM', 'EDU_MADRE_NUM', 'EDU_PROMEDIO_FAM','HORAS_TRABAJO_NUM','CATEGORIA_PROGRAMA'], drop_first=True,dtype=int)

#Creación de dataframe para modelos

In [49]:
# Definir las columnas a excluir (la variable target y su versión original)
drop_columns = ['RENDIMIENTO_GLOBAL', 'RENDIMIENTO_GLOBAL_encoded']

# 1. Conjunto de ENTRENAMIENTO (Base: df_model_2)
# X_train contiene todas las features de df_model_2, sin la target.
x_train = df_model_2.drop(columns=drop_columns)
# y_train contiene la variable target codificada.
y_train = df_model_2['RENDIMIENTO_GLOBAL_encoded']

# 2. Conjunto de PRUEBA (Base: df_test_model)
x_final_test = df_test_model.copy()

In [50]:
print("--- Resumen de los DataFrames ---")
print(f"Dimensiones de x_train: {x_train.shape}")
print(f"Dimensiones de x_final_test (sin y_test): {x_final_test.shape}")

--- Resumen de los DataFrames ---
Dimensiones de x_train: (692500, 82)
Dimensiones de x_final_test (sin y_test): (296786, 82)


Para este modelo se aplicó un preprocesamiento más avanzado, se crearon más variables a partir de las existentes con el fin de encontrar mejores relaciones entre variables que favorezcan al modelo, se crearon las variables:'AÑO','VALOR_MATRICULA_NUM', 'INDICE_SOCIOECO','REGION', 'F_ESTRATOVIVIENDA','E_PAGOMATRICULAPROPIO', 'ALGUNO_SUPERIOR','EDU_PADRE_NUM', 'EDU_MADRE_NUM', 'EDU_PROMEDIO_FAM','HORAS_TRABAJO_NUM','CATEGORIA_PROGRAMA', estas variables fueron explicadas anteriormente al momento de crearlas. Aquí se utilizará la librería Optuna para la optimización de hiperparámetros del clasificador CatBoost, se entrena el modelo final con la mejor configuración y se generan las predicciones sobre el conjunto de datos de prueba. Este proceso garantiza que el modelo final implementado sea el que ofrece el mejor rendimiento generalizable (el que tiene el Accuracy más alto en validación cruzada) antes de ser aplicado a datos reales sin etiqueta. CatBoost Classifier pertenece a la familia de algoritmos de Gradient Boosting. Su principal fortaleza radica en la capacidad de manejar las relaciones complejas y no lineales entre las features sin requerir pre-procesamiento avanzado, superando a modelos lineales más simples. Además, CatBoost es particularmente eficiente en el manejo de variables categóricas de alta cardinalidad y evita el sobreajuste mediante la técnica de "Ordered Boosting" y el uso de regularización, lo cual es vital para obtener el mejor Accuracy posible en un dataset donde las fronteras entre las clases (Alto, Medio-Alto, Medio-Bajo, Bajo) son inherentemente ambiguas.

#Modelo

##Optuna

En esta función se define un espacio de búsqueda donde se prueban diferentes configuraciones de hiperparámetros del modelo CatBoost (iterations, learning_rate, depth, etc.) utilizando los métodos de muestreo de Optuna (trial.suggest_int, trial.suggest_float). Esta función ejecuta el modelo con cada combinación de parámetros y utiliza la Validación Cruzada Estratificada para evaluar el rendimiento. El valor que retorna (np.mean(accuracy_scores)) es el Accuracy promedio de validación, que es la métrica que Optuna intenta maximizar para encontrar los parámetros que hacen que el modelo sea más robusto y generalizable.


In [51]:
def objective(trial):
    # 1. Definir el espacio de búsqueda de hiperparámetros
    param = {
        'iterations': trial.suggest_int('iterations', 200, 1000), # Número de árboles
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10), # Profundidad del árbol
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 100.0, log=True), # Regularización
        'subsample': trial.suggest_float('subsample', 0.6, 0.95), # Muestreo de datos
        'random_seed': 42,
        'verbose': 0,
        'loss_function': 'MultiClass',
        'eval_metric': 'Accuracy',
        'bootstrap_type': 'Bernoulli',
        'task_type': 'CPU'
    }

    # 2. Validación Cruzada Estratificada (para una evaluación robusta)
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    accuracy_scores = []

    for fold, (train_index, val_index) in enumerate(kf.split(x_train, y_train)):
        X_tr, X_val = x_train.iloc[train_index], x_train.iloc[val_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

        model = CatBoostClassifier(**param)
        model.fit(X_tr, y_tr, early_stopping_rounds=50) # Detención temprana

        y_pred = model.predict(X_val)
        y_pred_flat = y_pred.flatten() # CatBoost predice una matriz, se aplana

        accuracy_scores.append(accuracy_score(y_val, y_pred_flat))

    # 3. Optuna busca minimizar o maximizar el promedio de los scores de CV
    return np.mean(accuracy_scores)

Se utiliza la Validación Cruzada Estratificada para dividir el conjunto de entrenamiento en tres subconjuntos, garantizando que la proporción de las 4 clases de rendimiento sea la misma en cada subconjunto. En cada ciclo (o fold), el modelo CatBoost se entrena en dos subconjuntos (X_tr, y_tr) y se evalúa en el subconjunto restante (X_val, y_val). Además, se implementa la detención temprana (early_stopping_rounds=50) para evitar el sobreajuste y reducir el tiempo de entrenamiento. El Accuracy de la porción de validación es guardado y promediado al final de todos los folds, proporcionando una estimación confiable del rendimiento del modelo en datos que no ha visto.

Se inicia el estudio de optimización (study = optuna.create_study) indicando que el objetivo es maximizar el Accuracy. Al ejecutar study.optimize, Optuna realiza 5 pruebas (n_trials=5), utilizando las estrategias de muestreo para encontrar la mejor combinación de parámetros. Una vez finalizada la búsqueda, se extraen los mejores parámetros (study.best_params), los cuales se usan para inicializar el modelo CatBoost final. Este modelo final es entrenado por última vez con el 100% del conjunto de entrenamiento (x_train, y_train) para maximizar el uso de los datos en la configuración óptima.

In [52]:
# Crear el estudio de optimización, maximizar el Accuracy.
study = optuna.create_study(direction='maximize')

# n_trials: Número de combinaciones (pruebas) que Optuna debe realizar
study.optimize(objective, n_trials=5, show_progress_bar=True, n_jobs=1)

print("\n--- Resultados de la Optimización ---")
print(f"Mejor Accuracy de CV: {study.best_value:.4f}")
print("Mejores Parámetros:")
print(study.best_params)

# Obtener los mejores parámetros
best_catboost_params = study.best_params

[I 2025-11-28 00:09:30,504] A new study created in memory with name: no-name-5fe426ef-87b9-4e9e-809f-e5d1a25c1517


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2025-11-28 00:41:51,609] Trial 0 finished with value: 0.4077458478850309 and parameters: {'iterations': 751, 'learning_rate': 0.05742407007522656, 'depth': 10, 'l2_leaf_reg': 0.025307233994241604, 'subsample': 0.7781622918370615}. Best is trial 0 with value: 0.4077458478850309.
[I 2025-11-28 01:02:29,151] Trial 1 finished with value: 0.4021400713739105 and parameters: {'iterations': 527, 'learning_rate': 0.1821707315201539, 'depth': 9, 'l2_leaf_reg': 0.006013767925036917, 'subsample': 0.8626613465786362}. Best is trial 0 with value: 0.4077458478850309.
[I 2025-11-28 01:30:01,380] Trial 2 finished with value: 0.408945848047686 and parameters: {'iterations': 961, 'learning_rate': 0.05997321618924884, 'depth': 9, 'l2_leaf_reg': 2.3354251573421033e-06, 'subsample': 0.6025662536130171}. Best is trial 2 with value: 0.408945848047686.
[I 2025-11-28 01:51:29,848] Trial 3 finished with value: 0.40404043272673107 and parameters: {'iterations': 710, 'learning_rate': 0.12568132703923715, 'depth

Una vez entrenado el modelo final, se realizan dos pasos de verificación y predicción. Primero, se predice sobre el mismo conjunto de entrenamiento (x_train) para calcular el Accuracy de Entrenamiento. Este valor sirve como referencia de memorización para confirmar que el modelo ha aprendido eficientemente los datos, aunque no es un indicador de la capacidad de generalización. Segundo, y más importante, el modelo se aplica al conjunto de prueba sin etiqueta (x_final_test) para generar el array de resultados (y_pred_final_class)

In [53]:
# Modelo Final (CatBoost Clasificador MultiClass)
final_catboost_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=0,
    bootstrap_type='Bernoulli',
    **best_catboost_params
)

# Entrenamos con el 100%
final_catboost_model.fit(x_train, y_train)

# El clasificador ya predice la clase entera (0, 1, 2, 3)
y_pred_final_class = final_catboost_model.predict(x_final_test).flatten().astype(int)

df_predicciones = pd.DataFrame({
    'RENDIMIENTO_GLOBAL_PREDICHAS_ENCODED': y_pred_final_class,
})

print("Predicciones Finales generadas")
print(f"Número de predicciones generadas: {len(df_predicciones)}")

Predicciones Finales generadas
Número de predicciones generadas: 296786


In [54]:
y_pred_train_check = final_catboost_model.predict(x_train).flatten().astype(int)
accuracy_entrenamiento = accuracy_score(y_train, y_pred_train_check)
print(f"Accuracy en el conjunto de train: {accuracy_entrenamiento:.4f}")

Accuracy en el conjunto de train: 0.4958


La predicción numérica codificada (y_pred_final_class que contiene 0, 1, 2 o 3) se combina con el identificador único (ID) extraído del DataFrame de prueba para crear la tabla de resultados. Finalmente, se aplica un mapeo de decodificación (mapeo_decodificacion) que convierte las clases numéricas de vuelta a las etiquetas legibles (alto, medio-alto, medio-bajo, bajo). Esto genera el df_resultado_final en el formato solicitado, listo para la entrega y para cargarlo a kaggle

In [58]:

#Crear un DataFrame con la columna ID original y las predicciones codificadas
df_predicciones_encoded = pd.DataFrame({
    # Extraemos la columna 'ID'
    'ID': df_test1['ID'].values,
    'RENDIMIENTO_GLOBAL_ENCODED_PREDICHAS': y_pred_final_class
})

mapeo_decodificacion = {
    0: 'alto',
    1: 'medio-alto',
    2: 'medio-bajo',
    3: 'bajo'
}

df_predicciones_final = df_predicciones_encoded.copy()
df_predicciones_final['RENDIMIENTO_GLOBAL'] = df_predicciones_final['RENDIMIENTO_GLOBAL_ENCODED_PREDICHAS'].map(mapeo_decodificacion)

df_resultado_final = df_predicciones_final[['ID', 'RENDIMIENTO_GLOBAL']]

print(df_resultado_final.head())
print(f"Total de predicciones generadas: {len(df_resultado_final)}")

       ID RENDIMIENTO_GLOBAL
0  550236               bajo
1   98545         medio-alto
2  499179               alto
3  782980               bajo
4  785185               bajo
Total de predicciones generadas: 296786


In [59]:
df_resultado_final

,ID,RENDIMIENTO_GLOBAL
0,550236,bajo
1,98545,medio-alto
2,499179,alto
3,782980,bajo
4,785185,bajo
...,...,...
296781,496981,medio-bajo
296782,209415,alto
296783,239074,medio-alto
296784,963852,alto


In [57]:
df_resultado_final.to_csv('predicciones_rendimiento_final_optuna5.csv', index=False)

El accuracy en el conjunto de entrenamiento para este modelo es de 0.4958, este fue el último resultado que se subió a kaggle con un accuracy en test de 0.41